# Render AI reels on a cloud GPU (Google Colab)

1. **Runtime > Change runtime type > GPU** (a free T4 works; L4/A100 on Colab Pro are faster).
2. Run the cells top to bottom. Finished reels are copied to your Google Drive under `ai-reels-output/`.

Rough speed on a T4: a 9 s realistic reel at full 1080x1920 takes about 5-10 min; simple templates take 1-3 min.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the project
If the repo is **private**, create a GitHub token (Settings > Developer settings > Fine-grained tokens, read-only access to `ai-reels`) and paste it when asked.

In [ ]:
import getpass, os, subprocess
REPO = "swagata15/ai-reels"
if not os.path.exists("/content/ai-reels"):
    url = f"https://github.com/{REPO}.git"
    if subprocess.run(["git", "clone", "--depth", "1", url, "/content/ai-reels"]).returncode != 0:
        token = getpass.getpass("Repo is private - GitHub token: ")
        subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", "/content/ai-reels"], check=True)
else:
    subprocess.run(["git", "-C", "/content/ai-reels", "pull"], check=True)
%cd /content/ai-reels
!ls reels

## 2. Install Blender (about 1 min)

In [ ]:
BLENDER_VERSION = "5.0.1"
import os
if not os.path.exists("/content/blender/blender"):
    !apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 libxxf86vm1 libxfixes3 libgl1 > /dev/null
    !wget -q https://download.blender.org/release/Blender{BLENDER_VERSION[:3]}/blender-{BLENDER_VERSION}-linux-x64.tar.xz -O /content/blender.tar.xz
    !mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
!/content/blender/blender -b --version | head -1

## 3. Render
Pick which reels to render. `--engine cycles` uses the GPU (OptiX/CUDA) automatically.
Already-rendered reels are skipped, so you can add specs and run again.

In [ ]:
REELS = "reels/*.json"   # or e.g. "reels/claude_brain_surgery.json"
!python scripts/batch.py "{REELS}" --blender /content/blender/blender --engine cycles --post "--glow --grain"

## 4. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/ai-reels-output
!cp -v output/*/*_reel.mp4 output/*/*_cover.jpg /content/drive/MyDrive/ai-reels-output/